# Vectorized Constrained Affine Block Demo

This notebook demonstrates a single `nn.Module` that supports mixed per-feature slope constraints while consuming a `[batch, n_features]` tensor.

## What this covers

- One block for all linear features
- Mixed constraints in the same parameter vector
- One forward pass over a 2D batch tensor
- Per-feature contribution output for interpretability
- A short training loop showing constraints remain valid

In [ ]:
from pathlib import Path

import torch

from constrained_affine import ParallelConstrainedAffineBlock

torch.manual_seed(7)
print('torch version:', torch.__version__)
print('working directory:', Path.cwd())

## 1. Define mixed slope constraints

- Feature 0: `0 <= A_0 <= 1`
- Feature 1: unbounded
- Feature 2: `A_2 >= -1`
- Feature 3: fixed at `2`
- Feature 4: `A_4 <= 3`

In [ ]:
a_lower = [0.0, None, -1.0, 2.0, None]
a_upper = [1.0, None, None, 2.0, 3.0]

block = ParallelConstrainedAffineBlock(
    n_features=5,
    a_lower=a_lower,
    a_upper=a_upper,
    init_A=[0.3, 0.0, -0.25, 2.0, 1.5],
    use_feature_bias=False,
)

block

## 2. Run one forward pass

The block returns the total output, feature-level contributions, and the current constrained slope vector.

In [ ]:
x_lin = torch.randn(4, 5)
out = block(x_lin, return_contrib=True)

print('x_lin shape:', tuple(x_lin.shape))
print('y shape:', tuple(out['y'].shape))
print('contrib shape:', tuple(out['contrib'].shape))
print('A:', out['A'])
print('global bias:', block.global_bias)
print('y:\n', out['y'])
print('contrib:\n', out['contrib'])

## 3. Validate the constraints

This cell confirms the reparameterized slopes satisfy the required bounds before training.

In [ ]:
A = out['A']

checks = {
    'feature_0_in_[0,1]': bool((A[0] >= 0.0) and (A[0] <= 1.0)),
    'feature_1_unbounded': True,
    'feature_2_ge_-1': bool(A[2] >= -1.0),
    'feature_3_eq_2': bool(torch.isclose(A[3], torch.tensor(2.0))),
    'feature_4_le_3': bool(A[4] <= 3.0),
}
checks

## 4. Train for a few steps

The goal here is only to show that gradients flow and the constrained slopes remain valid after optimization.

In [ ]:
optimizer = torch.optim.Adam(block.parameters(), lr=0.05)
target = torch.tensor([[0.2], [1.1], [-0.3], [0.5]])

loss_history = []
for step in range(5):
    optimizer.zero_grad()
    pred = block(x_lin)
    loss = torch.mean((pred - target) ** 2)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    print(f'step={step} loss={loss.item():.6f}')

trained_A = block.get_A()
print('trained A:', trained_A)

In [ ]:
post_checks = {
    'feature_0_in_[0,1]': bool((trained_A[0] >= 0.0) and (trained_A[0] <= 1.0)),
    'feature_2_ge_-1': bool(trained_A[2] >= -1.0),
    'feature_3_eq_2': bool(torch.isclose(trained_A[3], torch.tensor(2.0))),
    'feature_4_le_3': bool(trained_A[4] <= 3.0),
}
post_checks

In [ ]:
summary = {
    'initial_A': out['A'].detach().tolist(),
    'trained_A': trained_A.detach().tolist(),
    'loss_history': loss_history,
}
summary